In [27]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OrdinalEncoder
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.metrics import fbeta_score, f1_score

import warnings
warnings.filterwarnings("ignore")

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/kagglex-bipoc-2022-2023-ml-foundation/Sample_submission.csv
/kaggle/input/competitions/kagglex-bipoc-2022-2023-ml-foundation/Train.csv
/kaggle/input/competitions/kagglex-bipoc-2022-2023-ml-foundation/Test.csv


## Configurations

In [28]:
INPUT_PATH = "/kaggle/input/competitions/kagglex-bipoc-2022-2023-ml-foundation"
TARGET = "TARGET"
N_SPLITS = 5
RANDOM_STATE = 42

## Load data

In [29]:
train = pd.read_csv(f"{INPUT_PATH}/Train.csv")
test = pd.read_csv(f"{INPUT_PATH}/Test.csv")

In [30]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (157509, 42)
Test shape: (67505, 41)


In [31]:
def get_column_groups(df):
    num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
    cat_cols = df.select_dtypes(include=["object"]).columns.tolist()
    return num_cols, cat_cols

In [32]:
NUM_COLS, CAT_COLS = get_column_groups(train.drop(columns=["ID", "TARGET"]))

## EDA

In [33]:
missing_values_features = pd.DataFrame(train.isna().sum() / train.shape[0])

missing_values_features.reset_index(inplace=True)
missing_values_features = missing_values_features.rename(columns = {'index':'feature',
                                                                     0: 'prct_missing'})
 
missing_values_features[missing_values_features.prct_missing > 0].sort_values(by='prct_missing', ascending=False)

,feature,prct_missing
25,MIGMTR1,0.490893
26,MIGMTR3,0.490893
27,MIGMTR4,0.490893
29,MIGSUN,0.490893
32,PEFNTVTY,0.041921
33,PEMNTVTY,0.038144
34,PENATVTY,0.022196
22,GRINST,0.004444


In [34]:
CAT_COLS_TO_IMPUTE = missing_values_features[missing_values_features.prct_missing > 0]['feature'].tolist()

## Preprocessing

In [35]:
def preprocess(df):
    df = df.copy()

    for col in CAT_COLS_TO_IMPUTE:
        df[col] = df[col].fillna('unknown')

    for col in CAT_COLS:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col])

    return df


In [36]:
train = preprocess(train)
test = preprocess(test)

In [37]:
X = train.drop(columns=['ID','TARGET'])
y = train.TARGET

X_test = test.drop(columns=['ID'])

In [38]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2, 
    stratify=y, 
    random_state=42
)

## Train model

In [39]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))


In [40]:
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    
    print(f"\n===== Fold {fold + 1} =====")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = lgb.LGBMClassifier(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=-1,
        random_state=RANDOM_STATE
    )
    
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric="auc",
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    # OOF predictions
    oof_preds[valid_idx] = model.predict(X_valid)
    
    # Test predictions (averaged later)
    test_preds += model.predict(X_test) / N_SPLITS
    
    fold_score = fbeta_score(y_valid, oof_preds[valid_idx], beta=0.5)
    print("Fold F-beta Score:", fold_score)


===== Fold 1 =====
[LightGBM] [Info] Number of positive: 10384, number of negative: 115623
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021350 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1382
[LightGBM] [Info] Number of data points in the train set: 126007, number of used features: 40
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.082408 -> initscore=-2.410069
[LightGBM] [Info] Start training from score -2.410069
Fold F-beta Score: 0.6617319679430098

===== Fold 2 =====
[LightGBM] [Info] Number of positive: 10384, number of negative: 115623
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.021112 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1385
[LightGBM] [Info] Number of da

In [41]:
cv_score = fbeta_score(y, oof_preds, beta=0.5)
print("\n====================")
print("Overall CV F-beta Score:", cv_score)
print("====================")


Overall CV F-beta Score: 0.6706722241929762


## Create submission

In [42]:
test_id_col = test.ID
y_pred_test = model.predict(X_test)

In [43]:
output = pd.DataFrame(test_id_col)
output['TARGET'] = y_pred_test

In [44]:
output.to_csv("predictions.csv", index=False)